# Mitosis-DenseViT — DenseNet121 → ViT-B/16 Hybrid

End-to-end training of the hybrid model (Tables 7, 10, 11). This notebook is a
thin runner: the real logic lives in the `mitosis_denseit/` package
(`config.py`, `data.py`, `model.py`, `train.py`, `evaluate.py`, `main.py`).

**Before running:** upload the `mitosis_denseit` folder to your Drive at
`/content/drive/MyDrive/Split_Folder/mitosis_denseit` (or edit the path in the
`%cd` cell below). Then run top-to-bottom.

## 0. Set Keras backend to JAX

Keras 3 can run on TensorFlow, JAX, or PyTorch. This sets it to **JAX**
for faster XLA-compiled training on the Colab GPU. This must run **before**
Keras or TensorFlow is imported anywhere else, so keep this as the very
first code cell.


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

# Optional but recommended: stop JAX from grabbing all GPU memory up front,
# which plays nicer with Colab's shared GPUs.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import keras
print("Keras backend:", keras.backend.backend())


## 1. Environment
TF-compatible transformers stack. Run once, then **restart the runtime** (Runtime → Restart session), then Run all again.

In [ ]:
!pip install "transformers<5.0" safetensors tf-keras -q
!pip install -U "keras>=3.0" -q

In [ ]:
import transformers, tensorflow as tf, jax, keras
print("transformers:", transformers.__version__, "| tensorflow:", tf.__version__)
print("keras:", keras.__version__, "| backend:", keras.backend.backend())
# GPU check — this model is heavy, you want a GPU
print("TF sees GPU:", tf.config.list_physical_devices('GPU'))
print("JAX sees GPU:", jax.devices())


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Go to the project folder
Edit this path if you put the `mitosis_denseit` folder elsewhere.

In [ ]:
%cd /content/drive/MyDrive/ViT-Base_Hybrid/ViT_DenseNet_Hybrid
!ls -la


## 3b. (Run once) Clear old checkpoints after a model/data code change

`train.py` now auto-resumes from `denseit_phase1.keras` / `denseit_finetune.keras`
if they exist, to survive Colab disconnects. That's what you want for a
*genuine* disconnect mid-run — but if you've changed `model.py` or `data.py`
in a way that changes what the model learns (e.g. the pretrained-ViT-weight
fix), resuming from an old checkpoint means keeping the OLD, unfixed weights.

Run this cell once after a code change like that, to start genuinely fresh.
Skip it if you're just resuming a run that disconnected.

In [ ]:
import os
from config import CFG

for tag in ["phase1", "finetune"]:
    p = os.path.join(CFG.OUTPUT_DIR, f"denseit_{tag}.keras")
    if os.path.exists(p):
        os.remove(p)
        print("Removed old checkpoint:", p)
    else:
        print("No existing checkpoint at:", p)

## 4. (Optional) Sanity-check config & data before the long run

Confirms your dataset paths resolve and the class labels map correctly.
Look for both `Positive_Mitosis` and `Negative_Mitosis` in the counts.

In [ ]:
from config import CFG
from data import get_dataframes

train_df, valid_df, test_df = get_dataframes()
print("train:", len(train_df), "| valid:", len(valid_df), "| test:", len(test_df))
print(train_df['label'].value_counts().to_dict())
print(train_df[['label','label_encoded']].drop_duplicates().to_dict('records'))


## 5. Build the model and print the summary

Confirms the architecture builds and shows the ~93.5M-param structure.
With the pretrained-ViT-weight fix, you should see
`[model] Loaded pretrained MLP weights for N encoder blocks.` — if you see a
skip/failure message instead, that's now a real issue worth investigating
(e.g. a network/download problem), not the silent no-op it used to be.

In [ ]:
from model import build_densevit
model = build_densevit(load_pretrained_vit=True)
model.summary()
print("Total params:", f"{model.count_params():,}")


## 6. Train (two-phase, Table 10)

Phase 1: DenseNet frozen warmup. Phase 2: fine-tune last DenseNet block + last
ViT blocks. This is the long step — every epoch runs all images through the full
model on the GPU. Checkpoints save to `denseit_outputs/` in your Drive, so a
disconnect won't lose the whole phase (best-effort resume — see `train.py`).

If you set `CFG.CACHE_DATASETS = True` in `config.py`, the first epoch will be
about as slow as before (it has to populate the cache), but every epoch after
that should be noticeably faster.

You can either call the full pipeline (`main.run()`), or run the pieces
manually as below for more control.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from data import make_dataset
from train import train

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(valid_df, training=False)
test_ds  = make_dataset(test_df,  training=False)

y = train_df.label_encoded.values
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print("class_weight:", class_weight)

history = train(model, train_ds, val_ds, class_weight=class_weight)


## 7. Curves + test-set evaluation
Sigmoid-correct thresholding. Lower `threshold` below 0.5 to trade precision for recall.

In [ ]:
from evaluate import plot_training_curves, evaluate

plot_training_curves(history)
metrics = evaluate(model, test_ds, test_df, threshold=0.5)
print("Final test metrics:", {k: round(v,4) for k,v in metrics.items()})


In [ ]:
import os

save_dir = "/content/drive/MyDrive/ViT-Base_Hybrid/ViT_DenseNet_Hybrid"
os.makedirs(save_dir, exist_ok=True)

# Save the full model (architecture + weights + optimizer state)
model.save(os.path.join(save_dir, "model.keras"))

print("Model saved to:", save_dir)

In [ ]:
model.save_weights(os.path.join(save_dir, "model.weights.h5"))

In [ ]:
import pickle

# NOTE: train() in train.py already returns a plain dict, not a Keras
# History object -- so this is `history`, not `history.history`.
with open(os.path.join(save_dir, "history.pkl"), "wb") as f:
    pickle.dump(history, f)

## Alternative: one-shot run

Instead of cells 6–7, you can run the whole pipeline in one call:

```python
import main
model, history, metrics = main.run()
```